# 06 Candidate Model Training

Train lightweight YOLO candidate architectures on the same four-class `exp_D_full_pipeline` dataset.


## Purpose

Notebook 06 is architecture triage only. Every candidate model sees the same dataset, same train settings, same online augmentation settings, and the same validation split.

The dataset already contains the four PPE classes:

- `0 = person`
- `1 = helmet`
- `2 = vest`
- `3 = cleaning_coverall`

The test split is checked for dataset completeness only. It is not used to rank or select candidate architectures.


## Setup

Find the `v2_pipeline` root, add it to `sys.path`, and import training/evaluation helpers from `src/training`.


In [1]:
from pathlib import Path
import os
import sys

# These environment defaults avoid common Windows/Jupyter crashes caused by
# duplicate OpenMP runtimes or too many nested CPU threads during torch import.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import pandas as pd
import yaml
from IPython.display import display


def find_v2_root(start: Path) -> Path:
    """Find the v2_pipeline root from the current working directory."""
    for candidate in (start, *start.parents):
        if candidate.name == "v2_pipeline" and (candidate / "src").exists():
            return candidate

        # Allows the notebook to run from the repository root in VS Code.
        nested = candidate / "ppe-detection" / "v2_pipeline"
        if nested.exists() and (nested / "src").exists():
            return nested

    raise RuntimeError("Could not find v2_pipeline root")


V2_ROOT = find_v2_root(Path.cwd().resolve())
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

from src.training.evaluate_yolo import (
    benchmark_model_latency,
    evaluate_candidate_model,
    rank_candidate_results,
)
from src.training.train_yolo import train_candidate_models

V2_ROOT

WindowsPath('C:/Github/smart-factory-safety-monitoring-system/ppe-detection/v2_pipeline')

The notebook imports orchestration helpers only. Ultralytics itself is imported inside the helper functions when training, validation, or benchmarking actually runs, so config and JSON checks can run without starting model work.

The OpenMP environment variables are set before training libraries load. This avoids a Windows kernel crash caused by duplicate OpenMP runtimes such as `libomp.dll` and `libiomp5md.dll` being initialized in the same process.


## Load Configuration

Load training settings, online augmentation settings, and class metadata.


In [2]:
training_config_path = V2_ROOT / "configs" / "training_config.yaml"
augmentation_config_path = V2_ROOT / "configs" / "augmentation_config.yaml"
class_names_path = V2_ROOT / "configs" / "class_names.yaml"

with training_config_path.open("r", encoding="utf-8") as file_handle:
    training_config = yaml.safe_load(file_handle)
with augmentation_config_path.open("r", encoding="utf-8") as file_handle:
    augmentation_config = yaml.safe_load(file_handle)
with class_names_path.open("r", encoding="utf-8") as file_handle:
    class_config = yaml.safe_load(file_handle)

# Notebook 06 always uses Experiment D for architecture triage. The ablation
# study comes later, after one architecture has been selected.
data_yaml = V2_ROOT / "data_exp_D_full_pipeline.yaml"
candidate_models = list(training_config["candidate_models"])
TRIAGE_EPOCHS = int(training_config.get("candidate_epochs", training_config["epochs"]))
class_names = {int(class_id): name for class_id, name in class_config["names"].items()}

config_summary = pd.DataFrame(
    [
        {"setting": "data_yaml", "value": str(data_yaml)},
        {"setting": "candidate_models", "value": candidate_models},
        {"setting": "triage_epochs", "value": TRIAGE_EPOCHS},
        {
            "setting": "selection_priority",
            "value": training_config.get("selection_priority", []),
        },
        {"setting": "classes", "value": class_names},
    ]
)
display(config_summary)

,setting,value
0,data_yaml,C:\Github\smart-factory-safety-monitoring-syst...
1,candidate_models,"[yolov8n.pt, yolov9t.pt, yolov10n.pt, yolo11n...."
2,triage_epochs,50
3,selection_priority,"[recall, map50, map50_95, fps, model_size]"
4,classes,"{0: 'person', 1: 'helmet', 2: 'vest', 3: 'clea..."


`candidate_epochs` is used for architecture triage when present. The original `epochs` value remains available for final training later, so Notebook 06 can be shorter and cheaper than the final retrain.


## Dataset Safety Check

Confirm the full-pipeline experiment YAML and referenced folders exist before training starts.


In [3]:
if not data_yaml.exists():
    raise FileNotFoundError(f"Dataset YAML not found: {data_yaml}")

with data_yaml.open("r", encoding="utf-8") as file_handle:
    data_config = yaml.safe_load(file_handle)

# Confirm the experiment YAML still matches the four-class PPE taxonomy before
# training. This catches stale YAML files created before cleaning_coverall was added.
if int(data_config.get("nc", -1)) != 4:
    raise ValueError(f"Expected nc=4 in {data_yaml}, found {data_config.get('nc')}")

yaml_names = {
    int(class_id): name for class_id, name in data_config.get("names", {}).items()
}
if yaml_names != class_names:
    raise ValueError(
        "Dataset YAML class names do not match configs/class_names.yaml: "
        f"{yaml_names} != {class_names}"
    )

# The YAML path is relative to v2_pipeline. Resolve it here only for existence
# checks; the training helper writes its own runtime YAML for Ultralytics.
dataset_root = V2_ROOT / data_config["path"]
dataset_checks = pd.DataFrame(
    [
        {
            "split": split,
            "path": str(dataset_root / data_config[split]),
            "exists": (dataset_root / data_config[split]).exists(),
        }
        for split in ["train", "val", "test"]
    ]
)
display(dataset_checks)

missing_paths = dataset_checks.loc[~dataset_checks["exists"], "path"].tolist()
if missing_paths:
    raise FileNotFoundError(f"Experiment dataset folders are missing: {missing_paths}")

,split,path,exists
0,train,C:\Github\smart-factory-safety-monitoring-syst...,True
1,val,C:\Github\smart-factory-safety-monitoring-syst...,True
2,test,C:\Github\smart-factory-safety-monitoring-syst...,True


The test path is checked only for dataset completeness. Candidate selection must use validation metrics only; the untouched test split is saved for final evaluation.


## Candidate Models

Display the candidate checkpoints loaded from `configs/training_config.yaml`.


In [4]:
candidate_table = pd.DataFrame(
    [
        {"order": index + 1, "model_name": model_name}
        for index, model_name in enumerate(candidate_models)
    ]
)
display(candidate_table)

,order,model_name
0,1,yolov8n.pt
1,2,yolov9t.pt
2,3,yolov10n.pt
3,4,yolo11n.pt
4,5,yolo12n.pt
5,6,yolo26n.pt


If a checkpoint is unavailable or unsupported by the installed Ultralytics version, the training helper records that model as failed and continues to the next candidate.


## Training Arguments

Build shared Ultralytics training arguments for every candidate model.


In [5]:
online_aug = augmentation_config["online_augmentation"]

# Set DRY_RUN=True to verify notebook wiring without training models.
DRY_RUN = False

# Keep this False for normal reruns. Existing run folders receive a numeric
# suffix instead of being overwritten.
OVERWRITE_EXISTING_RUNS = False

train_args = {
    "epochs": TRIAGE_EPOCHS,
    "imgsz": int(training_config["imgsz"]),
    "batch": int(training_config["batch"]),
    "device": training_config["device"],
    "workers": int(training_config["workers"]),
    "patience": int(training_config["patience"]),
    "seed": int(training_config["seed"]),
    "amp": bool(training_config.get("amp", False)),
    "hsv_h": float(online_aug["hsv_h"]),
    "hsv_s": float(online_aug["hsv_s"]),
    "hsv_v": float(online_aug["hsv_v"]),
    "degrees": float(online_aug["degrees"]),
    "translate": float(online_aug["translate"]),
    "scale": float(online_aug["scale"]),
    "perspective": float(online_aug["perspective"]),
    "fliplr": float(online_aug["fliplr"]),
    "flipud": float(online_aug["flipud"]),
    "mosaic": float(online_aug["mosaic"]),
    "mixup": float(online_aug["mixup"]),
    "close_mosaic": int(online_aug["close_mosaic"]),
    # Keep Ultralytics training plots off during triage to avoid many PNGs per
    # candidate run. CSV reports below are the source of truth for selection.
    "plots": False,
    "dry_run": DRY_RUN,
    "overwrite": OVERWRITE_EXISTING_RUNS,
}

display(
    pd.DataFrame(
        [{"argument": key, "value": value} for key, value in train_args.items()]
    )
)

,argument,value
0,epochs,50
1,imgsz,640
2,batch,24
3,device,0
4,workers,8
5,patience,20
6,seed,42
7,amp,True
8,hsv_h,0.015
9,hsv_s,0.3


Online augmentation is passed as training arguments, not encoded into the dataset YAML. This keeps `data_exp_D_full_pipeline.yaml` focused on data locations and class names.


## Train Candidate Models

Train every candidate on `exp_D_full_pipeline` and collect training-run validation metrics.


In [6]:
candidate_runs_dir = V2_ROOT / "runs" / "candidate_models"

# One failed checkpoint should not stop the whole architecture triage. The helper
# records failures in the returned table and continues to the next candidate.
candidate_results = train_candidate_models(
    data_yaml=data_yaml,
    model_names=candidate_models,
    output_dir=candidate_runs_dir,
    train_args=train_args,
    continue_on_error=True,
)

display(candidate_results)

New https://pypi.org/project/ultralytics/8.4.68 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.10.20 torch-2.13.0.dev20260517+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\runs\candidate_models\_data_exp_D_full_pipeline_resolved.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.3, hsv_v=0.35, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.

,model_name,run_name,status,run_dir,precision,recall,map50,map50_95,fitness,training_time_seconds,best_weights_path,last_weights_path,model_size_mb,error_message,notes
0,yolov8n.pt,YOLOv8_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.95112,0.87657,0.93791,0.64551,None,349.017,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.953714,,
1,yolov9t.pt,YOLOv9_Tiny_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.95160,0.89555,0.93130,0.64509,None,509.871,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.416635,,
2,yolov10n.pt,YOLOv10_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.89763,0.86430,0.91674,0.62798,None,439.561,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.480219,,
3,yolo11n.pt,YOLO11_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.91244,0.88159,0.92000,0.64068,None,353.432,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.212244,,
4,yolo12n.pt,YOLO12_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.94772,0.88195,0.93088,0.65260,None,473.730,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.256433,,
5,yolo26n.pt,YOLO26_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.90379,0.86325,0.91841,0.64093,None,432.349,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.132085,,


Runs are saved under `runs/candidate_models/`. When `OVERWRITE_EXISTING_RUNS=False`, reruns use a clear numeric suffix instead of overwriting an existing run folder.


## Optional Validation Refresh and Latency Benchmark

Optionally refresh validation metrics from best weights and estimate inference latency on validation images.


In [7]:
RUN_POST_TRAIN_VALIDATION = False
RUN_LATENCY_BENCHMARK = True
MAX_BENCHMARK_IMAGES = int(training_config.get("latency_max_images", 20))

val_images_dir = dataset_root / data_config["val"]

for row_index, row in candidate_results.iterrows():
    if row["status"] != "trained":
        continue

    best_weights = Path(str(row.get("best_weights_path", "")))

    # Training already validates each epoch. This optional refresh is off by
    # default to avoid an extra validation pass unless you explicitly need it.
    if RUN_POST_TRAIN_VALIDATION:
        eval_metrics = evaluate_candidate_model(
            weights_path=best_weights,
            data_yaml=data_yaml,
            imgsz=int(training_config["imgsz"]),
            device=training_config["device"],
        )
        for metric_name in ["precision", "recall", "map50", "map50_95", "fitness"]:
            if eval_metrics.get(metric_name) is not None:
                candidate_results.at[row_index, metric_name] = eval_metrics[metric_name]
        candidate_results.at[row_index, "eval_status"] = eval_metrics["status"]
        candidate_results.at[row_index, "eval_error_message"] = eval_metrics[
            "error_message"
        ]

    # Latency uses validation images only. It does not save prediction images;
    # it only records timing metrics in the candidate CSV reports.
    if RUN_LATENCY_BENCHMARK:
        latency_metrics = benchmark_model_latency(
            weights_path=best_weights,
            sample_images_dir=val_images_dir,
            imgsz=int(training_config["imgsz"]),
            device=training_config["device"],
            max_images=MAX_BENCHMARK_IMAGES,
        )
        for metric_name, metric_value in latency_metrics.items():
            candidate_results.at[row_index, metric_name] = metric_value

display(candidate_results)

,model_name,run_name,status,run_dir,precision,recall,map50,map50_95,fitness,training_time_seconds,best_weights_path,last_weights_path,model_size_mb,error_message,notes,latency_status,benchmark_images,avg_inference_ms,fps,latency_error
0,yolov8n.pt,YOLOv8_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.95112,0.87657,0.93791,0.64551,None,349.017,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.953714,,,benchmarked,20.0,72.752335,13.745263,
1,yolov9t.pt,YOLOv9_Tiny_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.95160,0.89555,0.93130,0.64509,None,509.871,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.416635,,,benchmarked,20.0,75.355760,13.270386,
2,yolov10n.pt,YOLOv10_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.89763,0.86430,0.91674,0.62798,None,439.561,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.480219,,,benchmarked,20.0,68.452000,14.608777,
3,yolo11n.pt,YOLO11_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.91244,0.88159,0.92000,0.64068,None,353.432,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.212244,,,benchmarked,20.0,69.467310,14.395260,
4,yolo12n.pt,YOLO12_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.94772,0.88195,0.93088,0.65260,None,473.730,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.256433,,,benchmarked,20.0,71.644015,13.957900,
5,yolo26n.pt,YOLO26_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.90379,0.86325,0.91841,0.64093,None,432.349,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.132085,,,benchmarked,20.0,69.296940,14.430652,


Latency benchmarking uses validation images only and is treated as an engineering signal after the validation metrics. If benchmarking fails on a machine, the error is recorded without changing the candidate training results.


## Save Candidate Reports

Save results, failures, and ranked candidates under `reports/training`.


In [8]:
reports_dir = V2_ROOT / "reports" / "training"
reports_dir.mkdir(parents=True, exist_ok=True)

ranked_results = rank_candidate_results(candidate_results)
failures_df = candidate_results.loc[candidate_results["status"].eq("failed")].copy()

# Keep Notebook 06 artifacts compact: one results table, one failures table,
# and one ranked table. The YOLO run folders contain the raw training outputs.
candidate_results.to_csv(reports_dir / "candidate_model_results.csv", index=False)
failures_df.to_csv(reports_dir / "candidate_model_failures.csv", index=False)
ranked_results.to_csv(reports_dir / "candidate_model_ranking.csv", index=False)

print(f"Saved candidate reports to: {reports_dir}")

Saved candidate reports to: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\training


The failures report may be empty. That is fine; it exists so unsupported checkpoint names or unavailable weights are still documented when they happen.


## Ranked Candidate Table

Rank candidates by recall, mAP50, mAP50-95, FPS, then model size.


In [9]:
display_columns = [
    "rank",
    "model_name",
    "run_name",
    "status",
    "recall",
    "map50",
    "map50_95",
    "fps",
    "avg_inference_ms",
    "model_size_mb",
    "rank_score",
    "error_message",
]
existing_columns = [
    column for column in display_columns if column in ranked_results.columns
]
display(ranked_results[existing_columns])

,rank,model_name,run_name,status,recall,map50,map50_95,fps,avg_inference_ms,model_size_mb,rank_score,error_message
0,1,yolov9t.pt,YOLOv9_Tiny_expD,trained,0.89555,0.93130,0.64509,13.270386,75.355760,4.416635,8.964820e+08,
1,2,yolo12n.pt,YOLO12_Nano_expD,trained,0.88195,0.93088,0.65260,13.957900,71.644015,5.256433,8.828815e+08,
2,3,yolo11n.pt,YOLO11_Nano_expD,trained,0.88159,0.92000,0.64068,14.395260,69.467310,5.212244,8.825107e+08,
3,4,yolov8n.pt,YOLOv8_Nano_expD,trained,0.87657,0.93791,0.64551,13.745263,72.752335,5.953714,8.775086e+08,
4,5,yolov10n.pt,YOLOv10_Nano_expD,trained,0.86430,0.91674,0.62798,14.608777,68.452000,5.480219,8.652174e+08,
5,6,yolo26n.pt,YOLO26_Nano_expD,trained,0.86325,0.91841,0.64093,14.430652,69.296940,5.132085,8.641691e+08,


The ranking table intentionally emphasizes recall first because missed PPE or role-uniform detections, including `cleaning_coverall`, are the highest-risk failure mode for the safety pipeline.


## Checklist Before Notebook 07

- `data_exp_D_full_pipeline.yaml` exists and reports `nc: 4`.
- Candidate runs completed or failures were recorded clearly.
- `candidate_model_ranking.csv` exists under `reports/training/`.
- The top successful model was selected using validation metrics only.
- The untouched test split was not used for model selection.
